In [ ]:
import json
import os
import requests
from folium import Map, TileLayer, GeoJson

endpoint = os.environ.get("TITILER_URL", "http://127.0.0.1:8082")
public_endpoint = os.environ.get("TITILER_PUBLIC_URL", "http://127.0.0.1:8082")

### Create Mosaic for the whole collection

In [ ]:
# create Mosaic
body = {
    "collections": ["facebook-population-density"],
}

register_resp = requests.post(
    f"{endpoint}/searches/register",
    json=body,
)
register_resp.raise_for_status()
response = register_resp.json()
print(response)

In [ ]:
m = Map(
    location=(0, 0),
    zoom_start=5
)

# Fetch Tilejson (we HAVE TO add the asset name)
search_id = response.get("id")
tilejson_url = response.get("url")
if tilejson_url is None:
    if search_id is None:
        raise KeyError(f"Search registration response has no id or url: {response}")
    tilejson_url = f"{endpoint}/searches/{search_id}/WebMercatorQuad/tilejson.json"

tilejson_resp = requests.get(
    tilejson_url,
    params={
        # Info to add to the tilejson response
        "minzoom": 4,
        "maxzoom": 12,
        # query parameter to add to the tile URL
        "assets": "cog",
        "rescale": "0,100",
        "colormap_name": "viridis",
    }
)
if tilejson_resp.status_code == 404 and search_id is not None:
    tilejson_url = f"{endpoint}/searches/{search_id}/tilejson.json"
    tilejson_resp = requests.get(
        tilejson_url,
        params={
            "minzoom": 4,
            "maxzoom": 12,
            "assets": "cog",
            "rescale": "0,100",
            "colormap_name": "viridis",
        }
    )
tilejson_resp.raise_for_status()
tj_resp = tilejson_resp.json()
tj_resp["tiles"] = [tile.replace(endpoint, public_endpoint) for tile in tj_resp["tiles"]]
print(tj_resp)

aod_layer = TileLayer(
    tiles=tj_resp["tiles"][0],
    attr="Mosaic",
    min_zoom=4,
    max_zoom=12,
    max_native_zoom=12,    
)
aod_layer.add_to(m)
m